In [6]:
from ROOT import TFile, TCanvas, kBlack, TPad, TGaxis, kRed, kBlue, kGray, gROOT, gStyle
import uuid 
%jsroot on

fileName_rel = 'electrons632.root'
fileName_ref = 'electrons.root'

histoPath="electronValidation"

def getHisto(file, path):
    t_path = file.Get(path)
    return t_path 

fileRoot_rel = TFile(fileName_rel)
fileRoot_ref = TFile(fileName_ref)
histo_relPath = getHisto(fileRoot_rel,histoPath)
histo_refPath = getHisto(fileRoot_ref,histoPath)

gStyle.SetOptStat(111110)

In [7]:
def check_histogram_type(hist,type):
    return hist.InheritsFrom(type)

def createHistoPicture(histo1,histo2,rescale=True,rebin=1):
    if (rebin>1):
        histo1.Rebin(rebin)
        histo2.Rebin(rebin)

    histo2.SetLineColor(2)
    normalOrder = True
    if (rescale):
        normalOrder = (histo1.GetBinContent(histo1.GetMaximumBin()))/histo1.Integral() >  (histo2.GetBinContent(histo2.GetMaximumBin()))/histo2.Integral()
    else:
        normalOrder = (histo1.GetBinContent(histo1.GetMaximumBin())) >  (histo2.GetBinContent(histo2.GetMaximumBin()))
    
    return histo1,histo2,normalOrder
   
def compareHisto(histoName, rescale=True, rebin=1):
    histo_rel = histo_relPath.Get(histoName)
    histo_ref = histo_refPath.Get(histoName)
    cnv = createHistoPicture(histo_ref,histo_rel,False,rebin)
    fileName=histoName+".png"
    cnv.Print(fileName)
    return cnv,histo_rel,histo_ref

def createRatio(h1, h2):
    h3 = h1.Clone("h3")
    h3.SetLineColor(kBlack)
    h3.SetMarkerStyle(21)
    h3.SetTitle("")
    h3.SetMinimum(0.8)
    h3.SetMaximum(1.35)
    # Set up plot for markers and errors
    h3.Sumw2()
    h3.SetStats(0)
    h3.Divide(h2)
 
    # Adjust y-axis settings
    y = h3.GetYaxis()
    y.SetTitle("Ratio")
    y.SetNdivisions(505)
    y.SetTitleSize(20)
    y.SetTitleFont(43)
    y.SetTitleOffset(1.55)
    y.SetLabelFont(43)
    y.SetLabelSize(15)
 
    # Adjust x-axis settings
    x = h3.GetXaxis()
    x.SetTitleSize(20)
    x.SetTitleFont(43)
    x.SetTitleOffset(4.0)
    x.SetLabelFont(43)
    x.SetLabelSize(15)
 
    return h3

def createCanvasPads():
    # Check for an existing canvas and delete it if necessary
    #if gROOT.FindObject("c"):
    #   print("Explicitely Deleting canvas ")
    #    gROOT.FindObject("c").Close()
    #    del c
    canvas_name = f"c1_{uuid.uuid4().hex}" 
    c = TCanvas(canvas_name,"c" ,800, 800)
    # Upper histogram plot is pad1
    pad1 = TPad("pad1", "pad1", 0, 0.3, 1, 1.0)
    pad1.SetBottomMargin(0)  # joins upper and lower plot
    pad1.SetGridx()
    pad1.Draw()
    # Lower ratio plot is pad2
    c.cd()  # returns to main canvas before defining pad2
    pad2 = TPad("pad2", "pad2", 0, 0.05, 1, 0.3)
    pad2.SetTopMargin(0)  # joins upper and lower plot
    pad2.SetBottomMargin(0.2)
    pad2.SetGridx()
    pad2.Draw()
 
    return c, pad1, pad2

""" def drawRatio(histoName):
    cnvR = TCanvas("canvasratio")
    histo_rel = histo_relPath.Get(histoName)
    histo_ref = histo_refPath.Get(histoName)
    h3=createRatio(histo_rel ,histo_ref)
    h3.Draw("ep")
    
    fileName=histoName+"Ratio.png"
    cnvR.Print(fileName)
    return cnvR,h3 """

 
def comparePlots(histoName,rescale=False, rebin=1):
    # create required parts
    h1rr = histo_relPath.Get(histoName)
    h2rr = histo_refPath.Get(histoName)
    print("Is TProfile "+ str(check_histogram_type(h1rr,"TProfile")))

    if check_histogram_type(h1rr,"TProfile"):
        h1r = h1rr.ProjectionX()
        h2r = h2rr.ProjectionX()
    else:   
        h1r = h1rr
        h2r = h2rr

    h1,h2,normalOrder = createHistoPicture(h1r,h2r,rescale,rebin)

    h3 = createRatio(h1, h2)
    c, pad1, pad2 = createCanvasPads()
    print("Normal ordering "+str(normalOrder))
     # draw everything
    pad1.cd()
    h1.SetLineColor(kRed)
    h1.SetStats(1)
    h1.SetLineStyle(0)
    h1.SetLineWidth(2)
    h2.SetLineColor(kBlue)
    h2.SetStats(1)

    # Ugly trick to get the two statBoxes
    h1.Draw()
    pad1.Update()
    statBox1 = h1.GetListOfFunctions().FindObject("stats")
    statBox1.SetName("statbox1")
    h2.Draw()
    pad1.Update()
    statBox2 = h2.GetListOfFunctions().FindObject("stats")
    statBox2.SetName("statbox2")
    h1.SetStats(0)
    h2.SetStats(0)

    if (normalOrder): 
        if (rescale):
            h1.DrawNormalized()
            h2.DrawNormalized("same")
        else:
            h1.Draw()
            h2.Draw("same")
    else:
        if (rescale):
            h2.DrawNormalized()
            h1.DrawNormalized("same")
        else:
            h2.Draw()
            h1.Draw("same")
   
    pad1.Update()
    
    statBox1.SetTextColor(kRed)    
    statBox1.SetBorderSize(2)
    statBox1.SetFillColor(kGray)
    statBox1.SetFillColorAlpha(kGray, 0.9) # https://root.cern.ch/doc/master/classTAttFill.html
    statBox1.SetY2NDC(0.995)
    statBox1.SetY1NDC(0.755)
    statBox1.SetX2NDC(0.995)
    statBox1.SetX1NDC(0.795)
   
    pad1.Update()
    c.Update()
    print(h1.GetListOfFunctions())
#    statBox2 = h2.GetListOfFunctions().FindObject("stats")

    statBox2.SetTextColor(kBlue)
    y1 = statBox1.GetY1NDC()
    y2 = statBox1.GetY2NDC()
    statBox2.SetY1NDC(2*y1-y2)
    statBox2.SetY2NDC(y1)
    statBox2.SetBorderSize(2)
    statBox2.SetFillColor(kGray)

    statBox2.SetFillColorAlpha(kGray, 0.9)
    statBox2.SetX2NDC(0.995)
    statBox2.SetX1NDC(0.795)

    
    # to avoid clipping the bottom zero, redraw a small axis
#    h1.GetYaxis().SetLabelSize(0.0)
    axis = TGaxis(-5, 20, -5, 220, 20, 220, 510, "")
    axis.SetLabelFont(43)
    axis.SetLabelSize(15)
    axis.Draw()
    pad2.cd()
    h3.Draw("ep")
    fileName=histoName+".png"
    c.Print(fileName)
    return c ,h3,h1,h2
 
#if __name__ == "__main__":
#    c,h=ratioplot("EoPvsEtaProfMax",False)
#    c.Update()    
#    c.Draw()

#histoName='dRPhoPFcand_Barrel_EtaR'
#histoName='dRPhoPFcand_Barrel_Edge'
#histoName='dRPhoPFcand_all'
#histoName='EtaPhotonsBarrel'
#histoName='EtaPhotonsCheckEdge'
#histoName='EoPvsEtaProfMax'
#canvas=compareHisto(histoName,'False')
#canvas.Draw()


#histosToPrint=[['dRPhoPFcand_Barrel_EtaR',False],['dRPhoPFcand_Barrel_Edge',True],['NeutralHadronEta',False]]

#for i in histosToPrint:
#    compareHisto(i[0],i[1])



In [8]:
from ROOT import gROOT
gROOT.Reset()
c1,h,h1c,h2c=comparePlots("EoPvsEtaProf",False)


Is TProfile True
Normal ordering False
{ @0x7ffee001bfa8 }


Warning in <TFile::Append>: Replacing existing TH1: EoPvsEtaProf_px (Potential memory leak).
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: png file EoPvsEtaProf.png has been created


In [9]:
c,h,h1c,h2c=comparePlots("EoPExt",False)
c.Draw()
c.Update()

Is TProfile False
Normal ordering False
{ @0x7ffee001bfa8 }


Info in <TCanvas::Print>: png file EoPExt.png has been created


In [10]:
c,h,h1c,h2c=comparePlots("EoPvsEtaProfMax",False)
c.Draw()
c.Update()

Is TProfile True
Normal ordering True
{ @0x7ffee001bfa8 }


Warning in <TFile::Append>: Replacing existing TH1: EoPvsEtaProfMax_px (Potential memory leak).
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: png file EoPvsEtaProfMax.png has been created
